# Setup

In [1]:
from utils.utils import set_gpu
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore", message=r"Some weights of .* were not initialized from the model checkpoint at .* and are newly initialized")
set_gpu(5)

Set GPU to 5


In [8]:
from utils.database.qdrant import QdrantIndexManager
from utils.llm import get_embedding_model, get_llm
from utils.retriever.uu.embedding_uu import get_embedding_uu

llm = get_llm("nvidia-llama3.1")
embed_model = get_embedding_model("BAAI/bge-m3")

vector_store = QdrantIndexManager(embed_model=embed_model, index_name="uu-all-512-token").load_vector_index()
retriever_uu = get_embedding_uu(vector_store=vector_store, llm=llm, embed_model=embed_model)

Loaded index 'uu-all-512-token'.


In [3]:
llm

ChatNVIDIA(base_url='http://localhost:8000/v1', model='meta/llama-3.1-8b-instruct', temperature=0.5, max_tokens=16000, top_p=0.5)

In [4]:
from utils.utils import load_from_pickle

testset = load_from_pickle('validated_testset')

In [5]:
testset2 = testset
i = 1
print(testset2["cleaned_question"][i])
print(testset2["ground_truth"][i])

What does the term 'perolehan suara yang lebih luas secara berjenjang' mean in the context of the presidential election, as defined in the Indonesian law on presidential and vice-presidential elections?
The term 'perolehan suara yang lebih luas secara berjenjang' refers to the candidate who wins in more provinces, as defined in the Indonesian law on presidential and vice-presidential elections.


# Helper Functions

In [15]:
from transformers import XLMRobertaTokenizer

tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-large")

In [16]:
import random
from itertools import zip_longest
from deep_translator import GoogleTranslator
from langdetect import detect

def split_to_chunks(text, chunk_size, chunk_overlap):
    # split context into chunks based on chunk_size and chunk_overlap
    tokens = tokenizer.encode(text)
    num_tokens = len(tokens)

    # print(f"Total tokens: {num_tokens}")
    
    chunks = []
    for j in range(0, num_tokens, chunk_size - chunk_overlap):
        chunk_tokens = tokens[j:j + chunk_size]
        
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True, max_length=None)
        chunks.append(chunk_text)
        
        if j + chunk_size >= num_tokens:
            break

    return chunks

def random_except(n, x):
    # return a random number between 0 and n except x
    n -= 1
    result = random.randint(0, n)
    while result == x:
        result = random.randint(0, n)
    return result

def alternate_chunks(chunk_a, chunk_b):
    # a = [1,2]
    # b = [3,4]
    # result = [1,3,2,4]
    return [x for pair in zip_longest(chunk_a, chunk_b) for x in pair if x is not None]

translator_en = GoogleTranslator(source='auto', target='en')

def translate_to_eng(query):
    # if indonesian, translate to english
    # otherwise, return original
    try:
        lang = detect(query)
        if lang == "id":
            return translator_en.translate(query)
        return query
    except Exception as e:
        return query

# Metrics

In [10]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from bert_score import score
from datasets import Dataset 
from ragas.metrics import answer_correctness, answer_similarity
from ragas import evaluate


# Assuming `a` is ground truth and `b` is the LLM output

def bleu(a, b):
    smooth_fn = SmoothingFunction().method1
    return sentence_bleu([a.split()], b.split(), smoothing_function=smooth_fn)

def rouge(a, b):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    return scorer.score(a, b)

def meteor(a, b):
    return meteor_score([a.split()], b.split())
    
def bert_score(a, b):
    P, R, F1 = score([b], [a], lang="en")
    return P, R, F1

def ragas(q, a, b):
    data_samples = {
        'question': [q],
        'answer': [b],
        'ground_truth': [a]
    }
    dataset = Dataset.from_dict(data_samples)
    score = evaluate(dataset, metrics=[answer_correctness, answer_similarity], llm=llm, embeddings=embed_model)
    return score

def evaluate_all_metrics(q, a, b):
    results = {}

    # General
    results['question'] = q
    results['ground_truth'] = a
    results['output'] = b

    # BLEU Score
    results['BLEU'] = bleu(a, b)
    
    # ROUGE Scores - capture precision, recall, and fmeasure for ROUGE-1, ROUGE-2, and ROUGE-L
    rouge_scores = rouge(a, b)
    results['ROUGE-1_Precision'] = rouge_scores['rouge1'].precision
    results['ROUGE-1_Recall'] = rouge_scores['rouge1'].recall
    results['ROUGE-1_F1'] = rouge_scores['rouge1'].fmeasure
    results['ROUGE-2_Precision'] = rouge_scores['rouge2'].precision
    results['ROUGE-2_Recall'] = rouge_scores['rouge2'].recall
    results['ROUGE-2_F1'] = rouge_scores['rouge2'].fmeasure
    results['ROUGE-L_Precision'] = rouge_scores['rougeL'].precision
    results['ROUGE-L_Recall'] = rouge_scores['rougeL'].recall
    results['ROUGE-L_F1'] = rouge_scores['rougeL'].fmeasure
    
    # METEOR Score
    results['METEOR'] = meteor(a, b)
    
    # BERTScore - capture individual precision, recall, and F1 scores
    P, R, F1 = bert_score(a, b)
    results['BERTScore_Precision'] = P.mean().item()  # Average over tokens
    results['BERTScore_Recall'] = R.mean().item()
    results['BERTScore_F1'] = F1.mean().item()

    # RAGAS metrics
    data_samples = {
        'question': [q],
        'answer': [b],
        'ground_truth': [a]
    }
    dataset = Dataset.from_dict(data_samples)
    # decided to omit answer_correctness
    ragas_scores = evaluate(dataset, metrics=[answer_similarity], llm=llm, embeddings=embed_model)
    
    # results['RAGAS_Answer_Correctness'] = ragas_scores['answer_correctness']
    results['RAGAS_Answer_Similarity'] = ragas_scores['answer_similarity']

    return results

# Baseline

In [ ]:
nrows = len(testset2)
questions = testset2["cleaned_question"].tolist()
contexts = testset2["contexts"].tolist()
ground_truths = testset2["ground_truth"].tolist()

In [10]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from utils.prompt.main import QA_SYSTEM_PROMPT_UU_COT
from utils.experiments.helpers import translate_to_eng

In [18]:
baseline = {}

for i in tqdm(range(nrows), desc=f"Processing baseline Rows"):
    # question
    question = questions[i]

    # ground truth
    ground_truth = translate_to_eng(ground_truths[i])

    # context
    context = contexts[i][0]

    bottom_context = [context]

    # prompt
    prompt = PromptTemplate.from_template(QA_SYSTEM_PROMPT_UU_COT)
    prompt = prompt.format(context=bottom_context, question=question)

    # llm
    response = llm.invoke(prompt).content

    # evaluate
    results = evaluate_all_metrics(question, ground_truth, response)

    # put to dict
    baseline[i] = results
    
    break

Processing baseline Rows:   0%|                                                                | 0/2644 [00:00<?, ?it/s]Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

Processing baseline Rows:   0%|                                                                | 0/2644 [00:04<?, ?it/s]


In [21]:
from utils.utils import save_to_pickle

save_to_pickle(baseline, 'baseline')

['DOC NAME: Undang-undang Nomor 42 Tahun 2008 Tentang Pemilihan Umum Presiden dan Wakil Presiden | UNDANG-UNDANG\n\n \n \n \n \nPRESIDEN \nREPUBLIK INDONESIA \n \n- 84 - \nd. mendapatkan akses informasi yang tersedia dari KPU, \nKPU provinsi, dan KPU kabupaten/kota; dan \ne. menggunakan perlengkapan untuk mendokumentasikan \nkegiatan \npemantauan \nsepanjang \nberkaitan \ndengan \npelaksanaan Pemilu Presiden dan Wakil Presiden.  \n \n(2) Pemantau asing yang berasal dari perwakilan negara asing \nyang berstatus diplomat berhak atas kekebalan diplomatik \nselama menjalankan tugas sebagai pemantau Pemilu Presiden \ndan Wakil Presiden. \n \nPasal 179 \n \nPemantau Pemilu Presiden dan Wakil Presiden mempunyai \nkewajiban: \n \na. mematuhi peraturan perundang-undangan dan menghormati \nkedaulatan Negara Kesatuan Republik Indonesia; \nb. mematuhi kode etik pemantau Pemilu Presiden dan Wakil \nPresiden yang diterbitkan oleh KPU; \nc. melaporkan diri, mengurus proses akreditasi dan tanda \npeng

# Context Order Sensitivity

In [11]:
CHUNK_SIZE = 512
CHUNK_OVERLAP = 128

In [12]:
nrows = len(testset2)
questions = testset2["cleaned_question"].tolist()
contexts = testset2["contexts"].tolist()
ground_truths = testset2["ground_truth"].tolist()

In [13]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from utils.prompt.main import QA_SYSTEM_PROMPT_UU_COT

## Bottom

In [14]:
cos_512_128_bottom = {}

for i in tqdm(range(nrows), desc="Processing rows"):
    # question
    question = questions[i]

    # ground truth
    ground_truth = translate_to_eng(ground_truths[i])

    # context
    random_i = random_except(nrows, i)

    context = contexts[i][0]
    noise_context = contexts[random_i][0]

    chunks = split_to_chunks(context, CHUNK_SIZE, CHUNK_OVERLAP)
    noise_chunks = split_to_chunks(noise_context, CHUNK_SIZE, CHUNK_OVERLAP)

    bottom_context = chunks + noise_chunks

    # prompt
    prompt = PromptTemplate.from_template(QA_SYSTEM_PROMPT_UU_COT)
    prompt = prompt.format(context=bottom_context, question=question)

    # llm
    response = llm.invoke(prompt).content

    # evaluate
    results = evaluate_all_metrics(question, ground_truth, response)

    # put to dict
    cos_512_128_bottom[i] = results
    
    break

Processing rows:   0%|                                                                         | 0/2644 [00:00<?, ?it/s]Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

Processing rows:   0%|                                                                         | 0/2644 [00:09<?, ?it/s]


In [ ]:
from utils.utils import save_to_pickle

save_to_pickle(cos_512_128_bottom, 'cos_512_128_bottom')

In [ ]:
load_from_pickle('cos_512_128_bottom')

## Alternate

In [ ]:
cos_512_128_alternate = {}

for i in tqdm(range(nrows), desc="Processing rows"):
    # question
    question = questions[i]

    # ground truth
    ground_truth = translate_to_eng(ground_truths[i])

    # context
    random_i = random_except(nrows, i)

    context = contexts[i][0]
    noise_context = contexts[random_i][0]

    chunks = split_to_chunks(context, CHUNK_SIZE, CHUNK_OVERLAP)
    noise_chunks = split_to_chunks(noise_context, CHUNK_SIZE, CHUNK_OVERLAP)

    bottom_context = alternate_chunks(chunks, noise_chunks)

    # prompt
    prompt = PromptTemplate.from_template(QA_SYSTEM_PROMPT_UU_COT)
    prompt = prompt.format(context=bottom_context, question=question)

    # llm
    response = llm.invoke(prompt).content

    # evaluate
    results = evaluate_all_metrics(question, ground_truth, response)

    # put to dict
    cos_512_128_alternate[i] = results

In [ ]:
from utils.utils import save_to_pickle

save_to_pickle(cos_512_128_alternate, 'cos_512_128_alternate')

In [ ]:
load_from_pickle('cos_512_128_alternate')

## Top 

In [ ]:
cos_512_128_top = {}

for i in tqdm(range(nrows), desc="Processing rows"):
    # question
    question = questions[i]

    # ground truth
    ground_truth = translate_to_eng(ground_truths[i])

    # context
    random_i = random_except(nrows, i)

    context = contexts[i][0]
    noise_context = contexts[random_i][0]

    chunks = split_to_chunks(context, CHUNK_SIZE, CHUNK_OVERLAP)
    noise_chunks = split_to_chunks(noise_context, CHUNK_SIZE, CHUNK_OVERLAP)

    bottom_context = chunks + noise_chunks

    # prompt
    prompt = PromptTemplate.from_template(QA_SYSTEM_PROMPT_UU_COT)
    prompt = prompt.format(context=bottom_context, question=question)

    # llm
    response = llm.invoke(prompt).content

    # evaluate
    results = evaluate_all_metrics(question, ground_truth, response)

    # put to dict
    cos_512_128_top[i] = results

In [ ]:
from utils.utils import save_to_pickle

save_to_pickle(cos_512_128_top, 'cos_512_128_top')

In [ ]:
load_from_pickle('cos_512_128_top')

## No Context

In [ ]:
cos_512_128_nocontext = {}

for i in tqdm(range(nrows), desc="Processing No Context Rows"):
    # question
    question = questions[i]

    # ground truth
    ground_truth = translate_to_eng(ground_truths[i])

    # context
    bottom_context = []

    # prompt
    prompt = PromptTemplate.from_template(QA_SYSTEM_PROMPT_UU_COT)
    prompt = prompt.format(context=bottom_context, question=question)

    # llm
    response = llm.invoke(prompt).content

    # evaluate
    results = evaluate_all_metrics(question, ground_truth, response)

    # put to dict
    cos_512_128_nocontext[i] = results

In [ ]:
from utils.utils import save_to_pickle

save_to_pickle(cos_512_128_nocontext, 'cos_512_128_nocontext')

# Faulty Context Injection

In [7]:
CHUNK_SIZE = 512
CHUNK_OVERLAP = 128

In [8]:
nrows = len(testset2)
questions = testset2["cleaned_question"].tolist()
contexts = testset2["contexts"].tolist()
ground_truths = testset2["ground_truth"].tolist()

In [9]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from utils.prompt.main import QA_SYSTEM_PROMPT_UU_COT
# from utils.experiments.metrics import evaluate_all_metrics
from utils.experiments.helpers import split_to_chunks, random_except, translate_to_eng
import random

## Single Noise

In [10]:
fci_512_128_single = {}

for i in tqdm(range(nrows), desc=f"Processing fci_{CHUNK_SIZE}_{CHUNK_OVERLAP}_single Rows"):
    # question
    question = questions[i]

    # ground truth
    ground_truth = translate_to_eng(ground_truths[i])

    # context
    random_i = random_except(nrows, i)

    context = contexts[i][0]
    noise_context = contexts[random_i][0]

    chunks = split_to_chunks(context, CHUNK_SIZE, CHUNK_OVERLAP)
    noise_chunks = split_to_chunks(noise_context, CHUNK_SIZE, CHUNK_OVERLAP)
    noise_chunk = [random.choice(noise_chunks)]

    bottom_context = chunks + noise_chunk
    random.shuffle(bottom_context)

    # prompt
    prompt = PromptTemplate.from_template(QA_SYSTEM_PROMPT_UU_COT)
    prompt = prompt.format(context=bottom_context, question=question)

    # llm
    response = llm.invoke(prompt).content

    # evaluate
    results = evaluate_all_metrics(question, ground_truth, response)

    # put to dict
    fci_512_128_single[i] = results

    break

Processing fci_512_128_single Rows:   0%|                                                      | 0/2644 [00:00<?, ?it/s]Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

Processing fci_512_128_single Rows:   0%|                                                      | 0/2644 [00:08<?, ?it/s]


In [ ]:
from utils.utils import save_to_pickle

save_to_pickle(fci_512_128_single, 'fci_512_128_single')

## Multiple Noise

In [57]:
fci_512_128_multiple = {}

for i in tqdm(range(nrows), desc=f"Processing fci_{CHUNK_SIZE}_{CHUNK_OVERLAP}_multiple Rows"):
    # question
    question = questions[i]

    # ground truth
    ground_truth = translate_to_eng(ground_truths[i])

    # context
    random_i = random_except(nrows, i)

    context = contexts[i][0]
    noise_context = contexts[random_i][0]

    chunks = split_to_chunks(context, CHUNK_SIZE, CHUNK_OVERLAP)
    noise_chunks = split_to_chunks(noise_context, CHUNK_SIZE, CHUNK_OVERLAP)
    num_noise = random.randint(1, len(noise_chunks))
    random_noise_chunks = random.sample(noise_chunks, num_noise)

    bottom_context = chunks + random_noise_chunks
    random.shuffle(bottom_context)

    # prompt
    prompt = PromptTemplate.from_template(QA_SYSTEM_PROMPT_UU_COT)
    prompt = prompt.format(context=bottom_context, question=question)

    # llm
    response = llm.invoke(prompt).content

    # evaluate
    results = evaluate_all_metrics(question, ground_truth, response)

    # put to dict
    fci_512_128_multiple[i] = results

    break

Processing fci_512_128_multiple Rows:   0%|                                                    | 0/2644 [00:00<?, ?it/s]Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

Processing fci_512_128_multiple Rows:   0%|                                                    | 0/2644 [00:05<?, ?it/s]


In [ ]:
from utils.utils import save_to_pickle

save_to_pickle(fci_512_128_multiple, 'fci_512_128_multiple')

## No Noise

In [59]:
fci_512_128_nonoise = {}

for i in tqdm(range(nrows), desc=f"Processing fci_{CHUNK_SIZE}_{CHUNK_OVERLAP}_nonoise Rows"):
    # question
    question = questions[i]

    # ground truth
    ground_truth = translate_to_eng(ground_truths[i])

    # context
    context = contexts[i][0]

    chunks = split_to_chunks(context, CHUNK_SIZE, CHUNK_OVERLAP)
    bottom_context = chunks
    random.shuffle(bottom_context)

    # prompt
    prompt = PromptTemplate.from_template(QA_SYSTEM_PROMPT_UU_COT)
    prompt = prompt.format(context=bottom_context, question=question)

    # llm
    response = llm.invoke(prompt).content

    # evaluate
    results = evaluate_all_metrics(question, ground_truth, response)

    # put to dict
    fci_512_128_nonoise[i] = results

    break

Processing fci_512_128_nonoise Rows:   0%|                                                     | 0/2644 [00:00<?, ?it/s]Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

Processing fci_512_128_nonoise Rows:   0%|                                                     | 0/2644 [00:05<?, ?it/s]


In [61]:
from utils.utils import save_to_pickle

save_to_pickle(fci_512_128_nonoise, 'fci_512_128_nonoise')

{0: {'question': 'What individuals or entities have the right to access information from the KPU, KPU provinsi, and KPU kabupaten/kota, according to the Undang-undang Nomor 42 Tahun 2008 Tentang Pemilihan Umum Presiden dan Wakil Presiden?',
  'ground_truth': 'Foreign observers who come from foreign representatives with diplomatic status are entitled to diplomatic immunity while carrying out their duties as observers of the Presidential and Vice Presidential Elections, and observers of the Presidential and Vice Presidential Elections who comply with laws and regulations and respect the sovereignty of the Unitary State of the Republic of Indonesia.',
  'output': '**Answer**: Individuals or entities that have the right to access information from the KPU, KPU provinsi, and KPU kabupaten/kota, according to the Undang-undang Nomor 42 Tahun 2008 Tentang Pemilihan Umum Presiden dan Wakil Presiden, include: a. Pemantau asing yang berasal dari perwakilan negara asing yang berstatus diplomat, b. 

# System Prompt

In [ ]:
CHUNK_SIZE = 512
CHUNK_OVERLAP = 128

In [ ]:
nrows = len(testset2)
questions = testset2["cleaned_question"].tolist()
contexts = testset2["contexts"].tolist()
ground_truths = testset2["ground_truth"].tolist()

In [27]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from utils.prompt.main import QA_SYSTEM_PROMPT_UU_COT, QA_SYSTEM_PROMPT_UU_ZEROSHOT, QA_SYSTEM_PROMPT_UU_FEWSHOT
from utils.experiments.helpers import split_to_chunks, random_except, translate_to_eng
import random

## Zero Shot

In [32]:
sp_zero = {}

for i in tqdm(range(nrows), desc=f"Processing zero shot Rows"):
    # question
    question = questions[i]

    # ground truth
    ground_truth = translate_to_eng(ground_truths[i])

    # context
    context = contexts[i][0]

    chunks = split_to_chunks(context, CHUNK_SIZE, CHUNK_OVERLAP)
    bottom_context = chunks

    # prompt
    prompt = PromptTemplate.from_template(QA_SYSTEM_PROMPT_UU_ZEROSHOT)
    prompt = prompt.format(context=bottom_context, question=question)

    # llm
    response = llm.invoke(prompt).content

    # evaluate
    results = evaluate_all_metrics(question, ground_truth, response)

    # put to dict
    sp_zero[i] = results
    
    break

Processing baseline Rows:   0%|                                                                                                                                                         | 0/2644 [00:00<?, ?it/s]Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

Processing baseline Rows:   0%|                                                                                                                                                         | 0/2644 [00:07<?, ?it/s]


In [36]:
from utils.utils import save_to_pickle

save_to_pickle(sp_zero, 'sp_zero')

{'question': 'What individuals or entities have the right to access information from the KPU, KPU provinsi, and KPU kabupaten/kota, according to the Undang-undang Nomor 42 Tahun 2008 Tentang Pemilihan Umum Presiden dan Wakil Presiden?',
 'ground_truth': 'Foreign observers who come from foreign representatives with diplomatic status are entitled to diplomatic immunity while carrying out their duties as observers of the Presidential and Vice Presidential Elections, and observers of the Presidential and Vice Presidential Elections who comply with laws and regulations and respect the sovereignty of the Unitary State of the Republic of Indonesia.',
 'output': 'Menurut Undang-undang Nomor 42 Tahun 2008 Tentang Pemilihan Umum Presiden dan Wakil Presiden, individu atau entitas yang berhak atas akses informasi dari KPU, KPU provinsi, dan KPU kabupaten/kota adalah pemantau Pemilu Presiden dan Wakil Presiden.\n\nPasal 178 ayat (1) Undang-undang Nomor 42 Tahun 2008 Tentang Pemilihan Umum Presiden 

## Few-Shot

In [ ]:
sp_few = {}

for i in tqdm(range(nrows), desc=f"Processing zero shot Rows"):
    # question
    question = questions[i]

    # ground truth
    ground_truth = translate_to_eng(ground_truths[i])

    # context
    context = contexts[i][0]

    chunks = split_to_chunks(context, CHUNK_SIZE, CHUNK_OVERLAP)
    bottom_context = chunks

    # prompt
    prompt = PromptTemplate.from_template(QA_SYSTEM_PROMPT_UU_FEWSHOT)
    prompt = prompt.format(context=bottom_context, question=question)

    # llm
    response = llm.invoke(prompt).content

    # evaluate
    results = evaluate_all_metrics(question, ground_truth, response)

    # put to dict
    sp_few[i] = results
    
    break

In [ ]:
from utils.utils import save_to_pickle

save_to_pickle(sp_few, 'sp_few')

## Chain of Thought

In [37]:
sp_cot = {}

for i in tqdm(range(nrows), desc=f"Processing cot Rows"):
    # question
    question = questions[i]

    # ground truth
    ground_truth = translate_to_eng(ground_truths[i])

    # context
    context = contexts[i][0]

    chunks = split_to_chunks(context, CHUNK_SIZE, CHUNK_OVERLAP)
    bottom_context = chunks

    # prompt
    prompt = PromptTemplate.from_template(QA_SYSTEM_PROMPT_UU_COT)
    prompt = prompt.format(context=bottom_context, question=question)

    # llm
    response = llm.invoke(prompt).content

    # evaluate
    results = evaluate_all_metrics(question, ground_truth, response)

    # put to dict
    sp_cot[i] = results
    
    break

In [ ]:
from utils.utils import save_to_pickle

save_to_pickle(sp_cot, 'sp_cot')

# LLM Temperature

In [40]:
CHUNK_SIZE = 512
CHUNK_OVERLAP = 128

In [41]:
nrows = len(testset2)
questions = testset2["cleaned_question"].tolist()
contexts = testset2["contexts"].tolist()
ground_truths = testset2["ground_truth"].tolist()

In [42]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from utils.prompt.main import QA_SYSTEM_PROMPT_UU_COT
from utils.experiments.helpers import split_to_chunks, random_except, translate_to_eng
import random

## Low

In [43]:
llm2 = get_llm("nvidia-llama3.1", temperature=0.1)

In [44]:
llmtemp_low = {}

for i in tqdm(range(nrows), desc=f"Processing llmtemp_low Rows"):
    # question
    question = questions[i]

    # ground truth
    ground_truth = translate_to_eng(ground_truths[i])

    # context
    context = contexts[i][0]

    chunks = split_to_chunks(context, CHUNK_SIZE, CHUNK_OVERLAP)
    bottom_context = chunks

    # prompt
    prompt = PromptTemplate.from_template(QA_SYSTEM_PROMPT_UU_COT)
    prompt = prompt.format(context=bottom_context, question=question)

    # llm
    response = llm2.invoke(prompt).content

    # evaluate
    results = evaluate_all_metrics(question, ground_truth, response)

    # put to dict
    llmtemp_low[i] = results
    
    break

Processing llmtemp_low Rows:   0%|                                                                                                                                                      | 0/2644 [00:00<?, ?it/s]Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

Processing llmtemp_low Rows:   0%|                                                                                                                                                      | 0/2644 [00:06<?, ?it/s]


In [ ]:
from utils.utils import save_to_pickle

save_to_pickle(llmtemp_low, 'llmtemp_low')

## Mid

In [ ]:
llm2 = get_llm("nvidia-llama3.1", temperature=0.6)

In [ ]:
llmtemp_mid = {}

for i in tqdm(range(nrows), desc=f"Processing llmtemp_mid Rows"):
    # question
    question = questions[i]

    # ground truth
    ground_truth = translate_to_eng(ground_truths[i])

    # context
    context = contexts[i][0]

    chunks = split_to_chunks(context, CHUNK_SIZE, CHUNK_OVERLAP)
    bottom_context = chunks

    # prompt
    prompt = PromptTemplate.from_template(QA_SYSTEM_PROMPT_UU_COT)
    prompt = prompt.format(context=bottom_context, question=question)

    # llm
    response = llm2.invoke(prompt).content

    # evaluate
    results = evaluate_all_metrics(question, ground_truth, response)

    # put to dict
    llmtemp_mid[i] = results
    
    break

In [ ]:
from utils.utils import save_to_pickle

save_to_pickle(llmtemp_mid, 'llmtemp_mid')

## High

In [ ]:
llm2 = get_llm("nvidia-llama3.1", temperature=0.9)

In [ ]:
llmtemp_high = {}

for i in tqdm(range(nrows), desc=f"Processing llmtemp_high Rows"):
    # question
    question = questions[i]

    # ground truth
    ground_truth = translate_to_eng(ground_truths[i])

    # context
    context = contexts[i][0]

    chunks = split_to_chunks(context, CHUNK_SIZE, CHUNK_OVERLAP)
    bottom_context = chunks

    # prompt
    prompt = PromptTemplate.from_template(QA_SYSTEM_PROMPT_UU_COT)
    prompt = prompt.format(context=bottom_context, question=question)

    # llm
    response = llm2.invoke(prompt).content

    # evaluate
    results = evaluate_all_metrics(question, ground_truth, response)

    # put to dict
    llmtemp_high[i] = results
    
    break

In [ ]:
from utils.utils import save_to_pickle

save_to_pickle(llmtemp_high, 'llmtemp_high')

# LLM Model

In [8]:
CHUNK_SIZE = 512
CHUNK_OVERLAP = 128

In [9]:
nrows = len(testset2)
questions = testset2["cleaned_question"].tolist()
contexts = testset2["contexts"].tolist()
ground_truths = testset2["ground_truth"].tolist()

In [10]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from utils.prompt.main import QA_SYSTEM_PROMPT_UU_COT
from utils.experiments.helpers import split_to_chunks, random_except, translate_to_eng
import random

## facebook/opt-1.3b

In [95]:
from transformers import pipeline

generator = pipeline('text-generation', model="facebook/opt-1.3b", device='cuda')

In [97]:
q = "What is the capital of jawa barat?"
# q = "What is 1+1?"
# q = "Calculate 1 + 1. The answer is"
generator(q, do_sample=True, max_length=50)

[{'generated_text': 'What is the capital of jawa barat?\nI read it as "How are Jawa girls"\nI read it as how are Jawa girls with capital "J"'}]

## Llama 3.2 3B

In [14]:
import torch
from transformers import pipeline

model_id = "meta-llama/Llama-3.2-3B-Instruct"
pipe = pipeline(
    "text-generation",
    model=model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [15]:
llmmodel_llama3b = {}

for i in tqdm(range(nrows), desc=f"Processing llmmodel_llama3b Rows"):
    # question
    question = questions[i]

    # ground truth
    ground_truth = translate_to_eng(ground_truths[i])

    # context
    context = contexts[i][0]

    chunks = split_to_chunks(context, CHUNK_SIZE, CHUNK_OVERLAP)
    bottom_context = chunks

    # prompt
    prompt = PromptTemplate.from_template(QA_SYSTEM_PROMPT_UU_COT)
    prompt = prompt.format(context=bottom_context, question=question)

    # llm
    messages = [
        {"role": "system", "content": prompt},
        {"role": "user", "content": question},
    ]
    
    outputs = pipe(
        messages,
        max_new_tokens=16000,
    )
    response = outputs[0]["generated_text"][-1]['content']

    # evaluate
    results = evaluate_all_metrics(question, ground_truth, response)

    # put to dict
    llmmodel_llama3b[i] = results
    
    break

Processing llmmodel_llama3b Rows:   0%|                                                        | 0/2644 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Processing llmmodel_llama3b Rows:   0%|                                                        | 0/2644 [00:04<?, ?it/s]


RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
llmmodel_llama3b

In [21]:
messages = [
    {"role": "system", "content": prompt},
    {"role": "user", "content": question},
]

outputs = pipe(
    messages,
    max_new_tokens=16000,
    temperature=0.5,
    top_p=0.5
)
response = outputs[0]["generated_text"][-1]['content']

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


In [22]:
response

'Pemantau asing yang berasal dari perwakilan negara asing yang berstatus diplomat berhak atas kekebalan diplomatik selama menjalankan tugas sebagai pemantau Pemilu Presiden dan Wakil Presiden.'

# SPTP

In [ ]:
shot_type_prompts = {
    "Zero": 'zero shot',
    "Few": 'few shot',
    "CoT": 'cot',
}

# Levels
levels = {
    "Low": 0.1,
    "Mid": 0.6,
    "High": 0.9
}

# Combinations
combinations = [
    ("Zero", "Low", "Low", "sptp_zll"),
    ("Few", "Low", "Low", "sptp_fll"),
    ("CoT", "Low", "Low", "sptp_cll"),
    ("Zero", "Mid", "Low", "sptp_zml"),
    ("Zero", "High", "Low", "sptp_zhl"),
    ("Zero", "Low", "Mid", "sptp_zlm"),
    ("Zero", "Low", "High", "sptp_zlh"),
    ("Zero", "Mid", "Mid", "sptp_zmm"),
    ("Zero", "Mid", "High", "sptp_zmh"),
    ("Zero", "High", "Mid", "sptp_zhm"),
    ("Zero", "High", "High", "sptp_zhh"),
    ("Few", "Mid", "Low", "sptp_fml"),
    ("Few", "High", "Low", "sptp_fhl"),
    ("Few", "Low", "Mid", "sptp_flm"),
    ("Few", "Low", "High", "sptp_flh"),
    ("Few", "Mid", "Mid", "sptp_fmm"),
    ("Few", "Mid", "High", "sptp_fmh"),
    ("Few", "High", "Mid", "sptp_fhm"),
    ("Few", "High", "High", "sptp_fhh"),
    ("CoT", "Mid", "Low", "sptp_cml"),
    ("CoT", "High", "Low", "sptp_chl"),
    ("CoT", "Low", "Mid", "sptp_clm"),
    ("CoT", "Low", "High", "sptp_clh"),
    ("CoT", "Mid", "Mid", "sptp_cmm"),
    ("CoT", "Mid", "High", "sptp_cmh"),
    ("CoT", "High", "Mid", "sptp_chm"),
    ("CoT", "High", "High", "sptp_chh"),
]

# Generate the dictionary
config = {}
for i, (shot_type, level1, level2, name) in enumerate(combinations):
    config[i] = {
        'sp': shot_type_prompts[shot_type],
        't': levels[level1],
        'p': levels[level2],
        'name': name
    }

In [ ]:
for k, v in config.items():
    print(k)
    print(v)
    sp = v['sp']
    t = v['t']
    p = v['p']
    name = v['name']

    print(sp, t, p, name)

    break
print(len(config))

# CS

In [111]:
import random

def no_context(chunks, noise_chunks):
    return []

def no_noise(chunks, noise_chunks):
    return chunks

def bottom_single(chunks, noise_chunks):
    noise_chunk = [random.choice(noise_chunks)]
    return chunks + noise_chunk

def bottom_multiple(chunks, noise_chunks):
    num_noise = random.randint(1, len(noise_chunks))
    random_noise_chunks = random.sample(noise_chunks, num_noise)
    return chunks + random_noise_chunks

def alternate_single(chunks, noise_chunks):
    noise_chunk = [random.choice(noise_chunks)]
    return alternate_chunks(chunks, noise_chunk)

def alternate_multiple(chunks, noise_chunks):
    num_noise = random.randint(1, len(noise_chunks))
    random_noise_chunks = random.sample(noise_chunks, num_noise)
    return alternate_chunks(chunks, random_noise_chunks)

def top_single(chunks, noise_chunks):
    noise_chunk = [random.choice(noise_chunks)]
    return noise_chunk + chunks

def top_multiple(chunks, noise_chunks):
    num_noise = random.randint(1, len(noise_chunks))
    random_noise_chunks = random.sample(noise_chunks, num_noise)
    return random_noise_chunks + chunks


In [120]:
a = [1, 2, 3]
b = [4, 5, 6, 7, 8, 9]
print("1.", no_context(a,b))
print("2.", no_noise(a,b))
print("3.", bottom_single(a, b))
print("4.", bottom_multiple(a, b))
print("5.", alternate_single(a, b))
print("6.", alternate_multiple(a, b))
print("7.", top_single(a, b))
print("8.", top_multiple(a, b))

1. []
2. [1, 2, 3]
3. [1, 2, 3, 8]
4. [1, 2, 3, 9, 7, 6, 4, 8, 5]
5. [1, 7, 2, 3]
6. [1, 8, 2, 9, 3, 6, 7, 4, 5]
7. [8, 1, 2, 3]
8. [6, 4, 8, 9, 7, 1, 2, 3]


In [132]:
# Strategies
strategies = {
    "No Context": no_context,
    "No Noise": no_noise,
    "Bottom Single": bottom_single,
    "Bottom Multiple": bottom_multiple,
    "Alternate Single": alternate_single,
    "Alternate Multiple": alternate_multiple,
    "Top Single": top_single,
    "Top Multiple": top_multiple
}

# Combinations
combinations = [
    ("No Context", "cs_nc"),
    ("No Noise", "cs_nn"),
    ("Bottom Single", "cs_bs"),
    ("Bottom Multiple", "cs_bm"),
    ("Alternate Single", "cs_as"),
    ("Alternate Multiple", "cs_am"),
    ("Top Single", "cs_ts"),
    ("Top Multiple", "cs_tm"),
]

# Generate the dictionary
config = {}
for i, (strategy, name) in enumerate(combinations):
    config[i] = {
        'strategy': strategies[strategy],
        'name': name
    }

In [133]:
config[0]

{'strategy': <function __main__.no_context(chunks, noise_chunks)>,
 'name': 'cs_nc'}

In [134]:
config[2]['strategy'](a, b)

[1, 2, 3, 4]

In [135]:
for k, v in config.items():
    print(k)
    print(v)
    strategy = v['strategy']
    name = v['name']

    print(strategy, name)

    break

0
{'strategy': <function no_context at 0x7fb93f624670>, 'name': 'cs_nc'}
<function no_context at 0x7fb93f624670> cs_nc


In [139]:
contexts[1][0]

'DOC NAME: Undang-undang Nomor 42 Tahun 2008 Tentang Pemilihan Umum Presiden dan Wakil Presiden | UNDANG-UNDANG\n\n \n \n \n \nPRESIDEN \nREPUBLIK INDONESIA \n \n- 31 - \n \nAyat (4)  \nYang dimaksud dengan “perolehan suara yang lebih luas secara \nberjenjang” adalah calon yang unggul di lebih banyak jumlah \nprovinsi, kabupaten/kota. \n \n          \nAyat (5)  \nCukup jelas. \n \nPasal 160 \n \nCukup jelas. \n \nPasal 161 \n \nCukup jelas. \n \nPasal 162 \n \nCukup jelas. \n \nPasal 163 \n \nCukup jelas. \n \nPasal 164 \n \nCukup jelas. \n \nPasal 165 \n \nCukup jelas. \n \nPasal 166 \n \nAyat (1) \n \n \n \nCukup jelas. \n \n \n \nAyat (2) \n \n \nHuruf a \nCukup jelas. \n \n \n \nHuruf b \nCukup jelas. \n \n \n \nHuruf c \nCukup jelas. \n \n \n \nHuruf d \nCukup jelas. \n \n \n \nHuruf e \nCukup jelas. \n \n \nHuruf f . . . \n\n'

# Context Chunking (Unrelated)

In [45]:
CHUNK_SIZE = 256
CHUNK_OVERLAP = 64

In [11]:
nrows = len(testset2)
questions = testset2["cleaned_question"].tolist()
contexts = testset2["contexts"].tolist()
ground_truths = testset2["ground_truth"].tolist()

In [41]:
# i = 2000

# context = contexts[i][0]
# context

In [42]:
context = max((inner[0] for inner in contexts), key=len)
context

'DOC NAME: Undang-undang Nomor 21 Tahun 2009 Tentang Pengesahan Agreement For The Implementation of The Provision of The United Nations Convention On The Law of The Sea of 10 Desember 1982 Relating to The Conservation and Management of Stradding Fish Stocks and Highly Migratory Fish Stock (persetujuan Pelaksanaan Ketentuan-ketentuan Konvensi Pbb Tentang Hukum Laut 1982 yang Berkaitan dengan Konservasi dan Pengelolaan Sediaan Ikan yang Beruaya Terbatas dan Sediaan Ikan yang Beruaya Jauh | UNDANG-UNDANG\n\n(a) with respect to straddling fish stocks, the relevant coastal States and the\nStates whose nationals fish for such stocks in the adjacent high seas area\nshall seek, either directly or through the appropriate mechanisms for\ncooperation provided for in Part III, to agree upon the measures necessary for\nthe conservation of these stocks in the adjacent high seas area;\n(b) with respect to highly migratory fish stocks, the relevant coastal States and\nother States whose nationals fish

In [46]:
chunks = split_to_chunks(context, CHUNK_SIZE, CHUNK_OVERLAP)
chunks

['DOC NAME: Undang-undang Nomor 21 Tahun 2009 Tentang Pengesahan Agreement For The Implementation of The Provision of The United Nations Convention On The Law of The Sea of 10 Desember 1982 Relating to The Conservation and Management of Stradding Fish Stocks and Highly Migratory Fish Stock (persetujuan Pelaksanaan Ketentuan-ketentuan Konvensi Pbb Tentang Hukum Laut 1982 yang Berkaitan dengan Konservasi dan Pengelolaan Sediaan Ikan yang Beruaya Terbatas dan Sediaan Ikan yang Beruaya Jauh | UNDANG-UNDANG (a) with respect to straddling fish stocks, the relevant coastal States and the States whose nationals fish for such stocks in the adjacent high seas area shall seek, either directly or through the appropriate mechanisms for cooperation provided for in Part III, to agree upon the measures necessary for the conservation of these stocks in the adjacent high seas area; (b) with respect to highly migratory fish stocks, the relevant coastal States and other States whose nationals fish for suc

In [39]:
len(chunks)

4